# 데이터안심구역 1차 집계 (2026-10-10, KDATA 서울센터)
주제: 고령운전자 면허 반납 후 이동 공백 지역 진단 및 DRT 우선 지역 도출
산출물: 사고·차량·말소 지역별 집계표 (반출 승인 6종)
원칙: 개별 레코드 반출 없음, 3건 미만 칸은 비공개 처리

In [ ]:
# 0. 환경
import os, sys
import pandas as pd
print(sys.version, pd.__version__)   # Python 3.10.14 / pandas 2.2.3

BASE = 'import_data/'
ACC  = BASE + 'TB_CKDATA_TRAFFIC_ACCIDENT/'        # 교통사고 2020~2024 (연도별 csv)
VEH  = BASE + 'TB_KTS_OPR_REG_WHOL_VHCL_INFO/'     # 운행등록 전체차량 (분기 스냅샷)
ERSR = BASE + 'TB_KTS_ERSR_REGIST/'                # 자동차 말소등록 (월별 csv)
MIN_N = 3                                          # 이 값 미만은 비공개

def mask_small(df, cols, n=MIN_N):
    # 소수 칸 비공개
    df = df.copy()
    for c in cols:
        df.loc[df[c] < n, c] = None
    return df

In [ ]:
# 0-1. 데이터 폴더 구조 확인 (폴더별 항목 수)
for d in sorted(os.listdir(BASE)):
    print(d, len(os.listdir(BASE + d)))

## 1. 교통사고 → 고령운전자(가해자 65세이상) 사고 집계

In [ ]:
# 1-1. 5개년 읽기 (필요 컬럼만)
cols = ['사고발생시도', '사고발생시군구', '사고발생읍면동', '사고발생년', '사고발생시', '가해자연령']
files = [x for x in os.listdir(ACC) if x.endswith('.csv')]
t = pd.concat([pd.read_csv(ACC + x, dtype=str, usecols=cols) for x in files])
print(t.shape)
print(t['사고발생년'].value_counts())   # 연 약 20만 건 (공식 통계와 일치)

In [ ]:
# 1-2. 고령 여부
t['고령'] = t['가해자연령'] == '65세이상'   # '불명' 제외됨

In [ ]:
# 1-3. 읍면동 / 시군구 집계
g = (t.groupby(['사고발생시도', '사고발생시군구', '사고발생읍면동'])
       .agg(전체=('고령', 'size'), 고령=('고령', 'sum')).reset_index())
s = (t.groupby(['사고발생시도', '사고발생시군구'])
       .agg(전체=('고령', 'size'), 고령=('고령', 'sum')).reset_index())
print(len(g), g['고령'].sum(), round((g['고령'] < MIN_N).mean(), 3))
# 읍면동 5,013개 / 고령 179,548건(약 18%) / 3건 미만 읍면동 14%

In [ ]:
# 1-4. 시간대별 · 연도별
old = t[t['고령']]
h = old.groupby(['사고발생시도', '사고발생시']).size().unstack(fill_value=0)
y = t.groupby(['사고발생시도', '사고발생년'])['고령'].agg(['size', 'sum']).reset_index()
print(old['사고발생시'].value_counts().sort_index())   # 08~18시 약 69%

In [ ]:
# 1-5. 저장 (반출)
mask_small(g, ['고령', '전체']).to_csv('acc_emd.csv', index=False, encoding='utf-8-sig')
s.to_csv('acc_sgg.csv', index=False, encoding='utf-8-sig')
h.to_csv('acc_hour.csv', encoding='utf-8-sig')
y.to_csv('acc_year.csv', index=False, encoding='utf-8-sig')

## 2. 운행등록 전체차량 → 시군구 × 소유자 연령대 × 차종
- 분기 스냅샷 8개 중 최신 202603만 사용 (중복 방지)
- 주소는 시군구까지만 제공 / 연령은 '70대 남자' 형식 → 앞 단어만 사용
- 약 74GB → 100만 행씩 분할 집계

In [ ]:
# 2-1. 구조 확인 (앞 5만 행)
f = [x for x in os.listdir(VEH) if '202603' in x][0]
v = pd.read_csv(VEH + f, nrows=50000, dtype=str)
print(v['회원등록번호'].value_counts().head())
print(v['사용본거지주소명'].str.split().str.len().value_counts())   # 2~3단어 = 시군구

In [ ]:
# 2-2. 분할 집계
cols = ['차종명', '회원구분명', '회원등록번호', '사용본거지주소명']
parts = []
for ch in pd.read_csv(VEH + f, dtype=str, usecols=cols, chunksize=1_000_000):
    ch['연령'] = ch['회원등록번호'].str.split().str[0]
    parts.append(ch.groupby(['사용본거지주소명', '회원구분명', '연령', '차종명'], dropna=False).size())
vv = (pd.concat(parts).groupby(level=[0, 1, 2, 3], dropna=False)
        .sum().reset_index(name='대수'))
print(vv.shape, vv['대수'].sum())   # 전국 2,653만 대

In [ ]:
# 2-3. 개인 소유 연령대 분포
print(vv[vv['회원구분명'] == '개인'].groupby('연령')['대수'].sum())
# 70대 215만 / 80대 37만 / 90대 4만 → 70대 이상 약 255만 대

In [ ]:
# 2-4. 저장 (반출)
mask_small(vv, ['대수']).to_csv('veh_sgg_202603.csv', index=False, encoding='utf-8-sig')

## 3. 말소등록 → 월 × 시군구 × 연령대 × 말소사유 (개인 소유)
- 말소사유: 신청구분명 (예: 자진말소(폐차))

In [ ]:
# 3-1. 월별 파일 집계
parts = []
for x in sorted(os.listdir(ERSR)):
    m = pd.read_csv(ERSR + x, dtype=str,
                    usecols=['회원구분명', '회원등록번호', '사용본거지주소명', '신청구분명'])
    m = m[m['회원구분명'] == '개인']
    m['연령'] = m['회원등록번호'].str.split().str[0]
    m['파일'] = x
    parts.append(m.groupby(['파일', '사용본거지주소명', '연령', '신청구분명'], dropna=False).size())
mm = pd.concat(parts).reset_index(name='건수')
print(mm.shape)

In [ ]:
# 3-2. 저장 (반출)
mask_small(mm, ['건수']).to_csv('ersr_sgg.csv', index=False, encoding='utf-8-sig')
# 메모: 월 단위라 칸의 39%가 비공개됨 → 연도 단위 재집계 검토

## 4. 정류장 목록 (센터 내부용, 반출 불가)
- 정류장 좌표는 정산사 영업정보 → 반출 반려됨 (10/12)
- 센터 안에서 읍면동 매핑 후 읍면동 집계만 반출할 예정

In [ ]:
# 4-1. 최신 월 · 첫날 기준 정류장별 경유 노선 수
RS = BASE + 'TB_KTS_ROUTESTTN/'
month = [x for x in sorted(os.listdir(RS)) if os.path.isdir(RS + x)][-1]   # 202609
f = sorted(os.listdir(RS + month))[0]                                      # 20260901
rs = pd.read_csv(RS + month + '/' + f, dtype=str)
st = (rs.groupby(['정류장ID', '정류장명칭', '정류장X좌표', '정류장Y좌표', '교통수단구분'])
        .agg(노선수=('노선ID', 'nunique')).reset_index())
st.to_csv('stop_list.csv', index=False, encoding='utf-8-sig')   # 내부용
print(st.shape)   # 188,676개